In [1]:
# 0. Import Libraries
import os
import shutil
import random
import cv2
import numpy as np
from tqdm import tqdm

In [2]:
# 1. Konfigurasi
source_dir = 'Rice_Leaf_AUG'
output_dir = 'Rice_Leaf_AUG_split'

# Rasio pembagian data (70% train, 20% val, 10% test)
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# Daftar kelas penyakit
classes = ['Bacterial Leaf Blight', 'Brown Spot', 'Healthy Rice Leaf', 'Leaf Blast']

In [3]:
# 2. Fungsi Grading
def calculate_grade(img_bgr):
    try:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

        lower_green = np.array([30, 40, 40]); upper_green = np.array([90, 255, 255])
        lower_dis1 = np.array([0, 40, 40]); upper_dis1 = np.array([30, 255, 255])
        lower_dis2 = np.array([150, 40, 40]); upper_dis2 = np.array([180, 255, 255])

        mask_green = cv2.inRange(hsv, lower_green, upper_green)
        mask_disease = cv2.inRange(hsv, lower_dis1, upper_dis1) + cv2.inRange(hsv, lower_dis2, upper_dis2)

        total = np.count_nonzero(mask_green) + np.count_nonzero(mask_disease)
        if total == 0: return 0
        
        ratio = (np.count_nonzero(mask_disease) / total) * 100
        
        if ratio == 0: return 0
        elif 0.1 <= ratio <= 10: return 1
        elif 10.1 <= ratio <= 25: return 2
        elif ratio > 25: return 3
        else: return 0
    except: return 0

In [4]:
# 3. Setup Folder Main
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

for split_name in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split_name, cls), exist_ok=True)

In [5]:
# 4. Proses Utama
for cls in classes:
    cls_dir = os.path.join(source_dir, cls)
    
    if not os.path.exists(cls_dir):
        print(f"Folder {cls_dir} tidak ditemukan, dilewat.")
        continue
        
    # Ambil SEMUA gambar, termasuk yang hasil augmentasi (aug_...)
    images = [f for f in os.listdir(cls_dir) 
              if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    
    random.seed(42)
    random.shuffle(images)

    total_images = len(images)
    train_count = int(total_images * train_ratio)
    val_count = int(total_images * val_ratio)

    train_images = images[:train_count]
    val_images = images[train_count:train_count + val_count]
    test_images = images[train_count + val_count:]

    def process_and_copy_files(file_list, split_name):
        count_saved = 0
        for file in tqdm(file_list, desc=f"   -> {split_name}", leave=False):
            src = os.path.join(cls_dir, file)
            
            # Khusus untuk baca gambar saat mau dilabelin grade
            img = cv2.imread(src)
            if img is None: continue
            
            # Hitung Grade Penyakit
            if cls == "Healthy Rice Leaf": 
                grade = 0
            else: 
                grade = calculate_grade(img)
            
            # Menyalin file saja (tanpa Resize/Augmentasi) tapi diganti Namanya pakai Grade_X
            dst_name = f"Grade_{grade}_{file}"
            dst = os.path.join(output_dir, split_name, cls, dst_name)
            
            shutil.copy(src, dst)
            count_saved += 1
            
        return count_saved

    print(f"\n🍃 Memproses Kelas: {cls}")
    
    t_count = process_and_copy_files(train_images, 'train')
    v_count = process_and_copy_files(val_images, 'val')
    test_count = process_and_copy_files(test_images, 'test')
    
    print(f"      Train: dari {len(train_images)} gambar -> {t_count} file disalin.")
    print(f"      Val  : dari {len(val_images)} gambar -> {v_count} file disalin.")
    print(f"      Test : dari {len(test_images)} gambar -> {test_count} file disalin.")

print("\n✅ SELESAI! Seluruh data (termasuk augmentasi asli) telah di-split dan di-labeli Grade.")


🍃 Memproses Kelas: Bacterial Leaf Blight


      Train: dari 445 gambar -> 445 file disalin.
      Val  : dari 127 gambar -> 127 file disalin.
      Test : dari 64 gambar -> 64 file disalin.

🍃 Memproses Kelas: Brown Spot


      Train: dari 452 gambar -> 452 file disalin.
      Val  : dari 129 gambar -> 129 file disalin.
      Test : dari 65 gambar -> 65 file disalin.

🍃 Memproses Kelas: Healthy Rice Leaf


      Train: dari 457 gambar -> 457 file disalin.
      Val  : dari 130 gambar -> 130 file disalin.
      Test : dari 66 gambar -> 66 file disalin.

🍃 Memproses Kelas: Leaf Blast


      Train: dari 443 gambar -> 443 file disalin.
      Val  : dari 126 gambar -> 126 file disalin.
      Test : dari 65 gambar -> 65 file disalin.

✅ SELESAI! Seluruh data (termasuk augmentasi asli) telah di-split dan di-labeli Grade.
